# Power Regulatory RAG — Week 1: PDF Ingestion Pipeline

## What we build today
```
Raw PDF → Page Classifier → Text/Table Extractor → Smart Chunker → Metadata Mapper → FAISS Store
```

Run each cell one by one. Every cell has a comment explaining **why** we do it, not just what.

---

## Cell 1 — Install dependencies
Run this once. Takes ~2 minutes.

In [ ]:
# Install all required libraries
# pymupdf      → fast PDF text extraction + page rendering
# pdfplumber   → better table extraction from digital PDFs
# pytesseract  → OCR for scanned pages
# Pillow       → image handling (needed for OCR)
# langchain    → chunking utilities
# sentence-transformers → free embedding model (no API key needed)
# faiss-cpu    → vector store
# python-dotenv → manage API keys

!pip install pymupdf pdfplumber pytesseract Pillow \
             langchain langchain-community \
             sentence-transformers faiss-cpu \
             python-dotenv openai -q

print("✅ All libraries installed")

## Cell 2 — Import libraries + verify
Always verify imports before going further.

In [ ]:
import fitz          # PyMuPDF
import pdfplumber
import json
import os
import io
from pathlib import Path
from PIL import Image

# Verify versions
print(f"PyMuPDF version  : {fitz.version}")
print(f"pdfplumber       : {pdfplumber.__version__}")
print("✅ Core imports OK")

## Cell 3 — Set your PDF path
Point this to whichever petition PDF you want to test with.

In [ ]:
# ⚠️  CHANGE THIS PATH to your actual petition PDF
PDF_PATH = "data/petitions/JUSNL_petition.pdf"

# Metadata you know about this document BEFORE parsing
# This is what we discussed — document-level metadata
DOC_METADATA = {
    "discom": "JUSNL",
    "document_type": "petition",   # petition / tariff_order / regulation
    "filing_year": "FY26",         # the year the petition covers
    "commission": "JSERC",
    "source_file": PDF_PATH
}

# Check file exists
if Path(PDF_PATH).exists():
    print(f"✅ PDF found: {PDF_PATH}")
    doc = fitz.open(PDF_PATH)
    print(f"   Total pages: {len(doc)}")
    doc.close()
else:
    print(f"⚠️  PDF not found at: {PDF_PATH}")
    print("   Place your PDF in the data/petitions/ folder and update PDF_PATH above")

---
## STAGE 1 — Page Classifier

**Why:** Before extracting anything, we need to know what kind of page we're dealing with.
- Digital page → PyMuPDF/pdfplumber works fine
- Scanned page → text = blank → need OCR

**How we detect:** If PyMuPDF extracts < 50 characters, the page is scanned.

In [ ]:
def detect_page_type(page: fitz.Page) -> str:
    """
    Classify a PDF page as 'digital' or 'scanned'.

    Digital  → PyMuPDF can extract readable text
    Scanned  → Page is an image, text extraction returns nothing
    """
    text = page.get_text().strip()

    # < 50 chars means either scanned or a blank/diagram page
    # Both cases need OCR to be useful
    if len(text) < 50:
        return "scanned"
    return "digital"


# ── TEST: Run classifier on first 10 pages of your PDF ──
if Path(PDF_PATH).exists():
    doc = fitz.open(PDF_PATH)
    print("Page | Type    | Chars extracted")
    print("-" * 40)
    for i in range(min(10, len(doc))):
        page = doc[i]
        ptype = detect_page_type(page)
        chars = len(page.get_text().strip())
        flag = "⚠️ " if ptype == "scanned" else "✅"
        print(f"  {i+1:2d} | {ptype:<8} | {chars:>5} chars  {flag}")
    doc.close()
else:
    print("⚠️  Add your PDF first (Cell 3)")

---
## STAGE 2 — Text Extractor (Digital Pages)

**Why pdfplumber over PyMuPDF for tables?**  
PyMuPDF is faster for raw text. pdfplumber is better at detecting table boundaries.  
We use both: PyMuPDF to classify, pdfplumber to extract.

In [ ]:
def table_to_text(table: list) -> str:
    """
    Convert raw pdfplumber table (nested list) → readable text.

    pdfplumber returns tables like:
        [['Cost Head', 'FY24 Approved', 'FY24 Actual'],
         ['Employee Cost', '120 Cr', '145 Cr'],
         ['Interest', '101 Cr', '540 Cr']]

    We convert to:
        Cost Head: Employee Cost | FY24 Approved: 120 Cr | FY24 Actual: 145 Cr
        Cost Head: Interest      | FY24 Approved: 101 Cr | FY24 Actual: 540 Cr

    This format is much easier for LLMs to understand than raw lists.
    """
    if not table or len(table) < 2:
        return ""

    # First row = headers
    headers = [str(h).strip() if h else f"col_{i}"
               for i, h in enumerate(table[0])]

    rows_text = []
    for row in table[1:]:  # skip header row
        if not any(row):   # skip completely empty rows
            continue
        # Build "Header: Value" pairs for each cell
        pairs = []
        for header, cell in zip(headers, row):
            cell_val = str(cell).strip() if cell else ""
            if cell_val:   # skip empty cells
                pairs.append(f"{header}: {cell_val}")
        if pairs:
            rows_text.append(" | ".join(pairs))

    return "\n".join(rows_text)


def extract_digital_page(pdf_path: str, page_num: int) -> dict:
    """
    Extract text and tables from a digital PDF page.
    Returns a dict with both raw text and cleaned table text.
    """
    with pdfplumber.open(pdf_path) as pdf:
        page = pdf.pages[page_num]
        raw_text = page.extract_text() or ""
        raw_tables = page.extract_tables() or []

    # Convert each table to readable text
    tables_as_text = []
    for table in raw_tables:
        text = table_to_text(table)
        if text:
            tables_as_text.append(text)

    return {
        "text": raw_text,
        "tables_text": tables_as_text,   # cleaned, LLM-readable
        "tables_raw": raw_tables,         # original nested list (for debugging)
        "page_number": page_num + 1,
        "page_type": "digital"
    }


# ── TEST: Extract a page that likely has a table ──
# Page 16 in JUSNL petition = Table 3 (Capex) — try this page
TEST_PAGE = 15   # 0-indexed → page 16 in PDF

if Path(PDF_PATH).exists():
    result = extract_digital_page(PDF_PATH, TEST_PAGE)

    print(f"=== Page {result['page_number']} ===")
    print(f"Text length  : {len(result['text'])} chars")
    print(f"Tables found : {len(result['tables_raw'])}")
    print()
    print("--- Text preview (first 400 chars) ---")
    print(result['text'][:400])
    print()
    if result['tables_text']:
        print("--- First table (cleaned for LLM) ---")
        print(result['tables_text'][0][:600])
    else:
        print("No tables found on this page — try a different TEST_PAGE number")
else:
    print("⚠️  Add your PDF first")

---
## STAGE 3 — OCR Extractor (Scanned Pages)

**Why:** Scanned pages are images. We render them at high DPI first, then run Tesseract OCR.

**Note:** Tesseract must be installed on your system separately:
- Windows: https://github.com/UB-Mannheim/tesseract/wiki
- Mac: `brew install tesseract`
- Linux: `sudo apt install tesseract-ocr`

In [ ]:
def extract_scanned_page(pdf_path: str, page_num: int) -> dict:
    """
    Extract text from a scanned PDF page using OCR.

    Steps:
    1. Render PDF page as high-res image (300 DPI)
    2. Run Tesseract OCR on the image
    3. Return extracted text

    Note: Tables from scanned pages come out as plain text.
    Structured table extraction from scans needs advanced tools
    like AWS Textract or Google Document AI.
    """
    try:
        import pytesseract
    except ImportError:
        return {"text": "", "tables_text": [], "tables_raw": [],
                "page_number": page_num + 1, "page_type": "scanned",
                "error": "pytesseract not installed"}

    doc = fitz.open(pdf_path)
    page = doc[page_num]

    # 300 DPI gives good OCR quality without being too slow
    # 72 DPI is screen resolution → 300/72 ≈ 4.16x scale
    mat = fitz.Matrix(300 / 72, 300 / 72)
    pix = page.get_pixmap(matrix=mat)
    img = Image.open(io.BytesIO(pix.tobytes("png")))
    doc.close()

    # Run OCR — 'eng' for English, add '+hin' if Hindi text exists
    text = pytesseract.image_to_string(img, lang='eng')

    return {
        "text": text,
        "tables_text": [],    # no structured tables from OCR
        "tables_raw": [],
        "page_number": page_num + 1,
        "page_type": "scanned"
    }


# ── TEST: Only run if you have a scanned page ──
# Find a scanned page number from Stage 1 output above
# then set SCANNED_PAGE to that number (0-indexed)

SCANNED_PAGE = None   # Set this to a scanned page number, e.g. 5

if SCANNED_PAGE is not None and Path(PDF_PATH).exists():
    result = extract_scanned_page(PDF_PATH, SCANNED_PAGE)
    print(f"=== Scanned Page {result['page_number']} ===")
    print(f"OCR text length: {len(result['text'])} chars")
    print("--- OCR output preview ---")
    print(result['text'][:400])
else:
    print("ℹ️  Skipped OCR test — set SCANNED_PAGE to a scanned page number")
    print("   (Run Stage 1 classifier first to find scanned pages)")

---
## STAGE 4 — Full PDF Loader

Combines classifier + both extractors. This is your main entry point.

In [ ]:
def load_pdf(pdf_path: str, doc_metadata: dict, max_pages: int = None) -> list[dict]:
    """
    Load entire PDF. For each page:
    1. Classify (digital or scanned)
    2. Extract with correct extractor
    3. Attach document-level metadata to every page

    Returns list of page dicts, one per page.
    """
    doc = fitz.open(pdf_path)
    total_pages = len(doc)

    # Optionally limit pages for testing
    pages_to_process = min(max_pages, total_pages) if max_pages else total_pages

    print(f"📄 Loading: {pdf_path}")
    print(f"   Total pages: {total_pages} | Processing: {pages_to_process}")
    print()

    all_pages = []

    for i in range(pages_to_process):
        page = doc[i]
        page_type = detect_page_type(page)

        # Route to correct extractor
        if page_type == "digital":
            page_data = extract_digital_page(pdf_path, i)
        else:
            page_data = extract_scanned_page(pdf_path, i)

        # Merge document-level metadata into every page
        # This is critical — every chunk inherits this later
        page_data.update(doc_metadata)

        all_pages.append(page_data)

        # Progress indicator
        icon = "✅" if page_type == "digital" else "⚠️ "
        print(f"  {icon} Page {i+1:3d}/{pages_to_process} | {page_type:<8} | "
              f"{len(page_data['text']):>5} chars | "
              f"{len(page_data.get('tables_raw', []))} tables")

    doc.close()
    print(f"\n✅ Done — {len(all_pages)} pages loaded")
    return all_pages


# ── TEST: Load first 20 pages (fast) ──
if Path(PDF_PATH).exists():
    pages = load_pdf(PDF_PATH, DOC_METADATA, max_pages=20)
    print(f"\nSample page keys: {list(pages[0].keys())}")
else:
    print("⚠️  Add your PDF first")

---
## STAGE 5 — Smart Chunker

**Why NOT fixed-size chunks:**  
A 500-token window can split a table across two chunks.  
The LLM then sees half a table and hallucinates the rest.

**Our strategy:**  
- Text → split by heading/section boundaries  
- Tables → each table = one dedicated chunk (never split a table)

In [ ]:
import re

# Headings found in Indian regulatory documents
# These signal a new section → natural chunk boundary
SECTION_PATTERNS = [
    r'^\d+\.\d+\s+[A-Z]',          # 3.3 Capital Expenditure
    r'^\d+\.\s+[A-Z]',             # 3. Overall Approach
    r'^[A-Z][A-Z\s]{10,}$',        # ALL CAPS HEADING
    r'^(Chapter|Section|Part)\s+',  # Chapter / Section / Part
]

def is_section_heading(line: str) -> bool:
    """Check if a line looks like a section heading."""
    line = line.strip()
    if len(line) < 5 or len(line) > 120:
        return False
    return any(re.match(p, line) for p in SECTION_PATTERNS)


def chunk_text(text: str, page_metadata: dict) -> list[dict]:
    """
    Split page text into chunks at section boundaries.
    Each chunk inherits the page's metadata.
    """
    lines = text.split('\n')
    chunks = []
    current_chunk_lines = []
    current_heading = "introduction"

    for line in lines:
        if is_section_heading(line):
            # Save current chunk before starting new section
            if current_chunk_lines:
                chunk_text = '\n'.join(current_chunk_lines).strip()
                if len(chunk_text) > 100:  # skip tiny fragments
                    chunks.append({
                        "text": chunk_text,
                        "chunk_type": "text",
                        "section_heading": current_heading,
                        **page_metadata   # inherit all page metadata
                    })
            # Start new section
            current_heading = line.strip()
            current_chunk_lines = [line]
        else:
            current_chunk_lines.append(line)

    # Don't forget the last chunk
    if current_chunk_lines:
        chunk_text_val = '\n'.join(current_chunk_lines).strip()
        if len(chunk_text_val) > 100:
            chunks.append({
                "text": chunk_text_val,
                "chunk_type": "text",
                "section_heading": current_heading,
                **page_metadata
            })

    return chunks


def chunk_tables(tables_text: list[str], page_metadata: dict) -> list[dict]:
    """
    Each table becomes its own dedicated chunk.
    Tables are NEVER split — they lose meaning if split.
    """
    chunks = []
    for i, table_text in enumerate(tables_text):
        if table_text.strip():
            chunks.append({
                "text": table_text,
                "chunk_type": "table",      # important for retrieval later
                "table_index": i,
                "section_heading": "table",
                **page_metadata
            })
    return chunks


def chunk_page(page_data: dict) -> list[dict]:
    """
    Chunk a single page's content.
    Returns list of chunks (text chunks + table chunks).
    """
    # Metadata to pass down to every chunk from this page
    page_metadata = {
        "page_number" : page_data["page_number"],
        "page_type"   : page_data["page_type"],
        "discom"      : page_data.get("discom", ""),
        "document_type": page_data.get("document_type", ""),
        "filing_year" : page_data.get("filing_year", ""),
        "commission"  : page_data.get("commission", ""),
        "source_file" : page_data.get("source_file", "")
    }

    all_chunks = []

    # Chunk the text
    if page_data.get("text"):
        text_chunks = chunk_text(page_data["text"], page_metadata)
        all_chunks.extend(text_chunks)

    # Chunk the tables (each table = one chunk)
    if page_data.get("tables_text"):
        table_chunks = chunk_tables(page_data["tables_text"], page_metadata)
        all_chunks.extend(table_chunks)

    return all_chunks


# ── TEST: Chunk all loaded pages ──
if 'pages' in dir() and pages:
    all_chunks = []
    for page_data in pages:
        chunks = chunk_page(page_data)
        all_chunks.extend(chunks)

    # Summary
    text_chunks  = [c for c in all_chunks if c['chunk_type'] == 'text']
    table_chunks = [c for c in all_chunks if c['chunk_type'] == 'table']

    print(f"Total chunks  : {len(all_chunks)}")
    print(f"Text chunks   : {len(text_chunks)}")
    print(f"Table chunks  : {len(table_chunks)}")
    print()
    print("--- Sample text chunk ---")
    if text_chunks:
        c = text_chunks[0]
        print(f"  Page       : {c['page_number']}")
        print(f"  Heading    : {c['section_heading']}")
        print(f"  Discom     : {c['discom']}")
        print(f"  Text preview: {c['text'][:200]}")
    print()
    print("--- Sample table chunk ---")
    if table_chunks:
        c = table_chunks[0]
        print(f"  Page       : {c['page_number']}")
        print(f"  Table index: {c['table_index']}")
        print(f"  Content    : {c['text'][:300]}")
else:
    print("⚠️  Load PDF first (Stage 4)")

---
## STAGE 6 — LLM Metadata Mapper

**Why:** Section headings vary across documents.  
JUSNL says "3.3 Capital Expenditure", JBVNL might say "2.1 Capex Details".  
We use an LLM to map both → `cost_head: capex`

**Standard cost heads (from regulation):**
```
capex, employee_cost, om_expenses, depreciation, 
interest, roe, power_purchase, revenue_gap, tariff, other
```

In [ ]:
# Option A: Use OpenAI (needs API key)
# Option B: Use a free local approach with keyword matching first,
#           then LLM only for ambiguous cases
#
# We start with Option B — no API key needed to learn

# Standard cost heads from Indian electricity regulations
COST_HEAD_KEYWORDS = {
    "capex"         : ["capital expenditure", "capex", "cwip", "capitalization",
                       "capital investment", "scheme", "project"],
    "employee_cost" : ["employee", "salary", "staff", "manpower", "hr",
                       "wages", "pension", "gratuity"],
    "om_expenses"   : ["operation and maintenance", "o&m", "repair",
                       "maintenance", "a&g", "administrative"],
    "depreciation"  : ["depreciation", "asset life", "useful life", "dep rate"],
    "interest"      : ["interest", "loan", "borrowing", "debt", "working capital",
                       "interest on loan"],
    "roe"           : ["return on equity", "roe", "equity", "normative equity"],
    "power_purchase": ["power purchase", "ppc", "energy charge", "capacity charge",
                       "ppa", "merit order", "short term purchase"],
    "revenue_gap"   : ["revenue gap", "regulatory asset", "carrying cost",
                       "unrecovered", "revenue requirement"],
    "tariff"        : ["tariff", "consumer category", "slab", "transmission charge",
                       "wheeling charge"],
    "true_up"       : ["true up", "true-up", "trueup", "actual vs approved",
                       "reconciliation", "provisional"],
    "arr"           : ["annual revenue requirement", "arr", "aggregate revenue"],
    "at_c_loss"     : ["at&c", "aggregate technical", "commercial loss", "transmission loss"]
}

def map_cost_head_keywords(text: str, heading: str) -> str:
    """
    Fast keyword-based cost head mapping.
    Check heading first (more reliable), then full text.
    Returns matched cost head or 'other'.
    """
    search_text = (heading + " " + text[:500]).lower()

    scores = {}
    for cost_head, keywords in COST_HEAD_KEYWORDS.items():
        score = sum(1 for kw in keywords if kw in search_text)
        if score > 0:
            scores[cost_head] = score

    if scores:
        return max(scores, key=scores.get)  # return highest scoring cost head
    return "other"


def add_metadata_to_chunks(chunks: list[dict]) -> list[dict]:
    """
    Add cost_head to every chunk.
    Uses keyword matching (fast, free).
    Later we'll upgrade this to LLM mapping for ambiguous cases.
    """
    for chunk in chunks:
        cost_head = map_cost_head_keywords(
            chunk.get("text", ""),
            chunk.get("section_heading", "")
        )
        chunk["cost_head"] = cost_head
    return chunks


# ── TEST: Add cost_head metadata to all chunks ──
if 'all_chunks' in dir() and all_chunks:
    all_chunks = add_metadata_to_chunks(all_chunks)

    # Show distribution of cost heads
    from collections import Counter
    cost_head_counts = Counter(c['cost_head'] for c in all_chunks)

    print("Cost head distribution across chunks:")
    print("-" * 35)
    for cost_head, count in cost_head_counts.most_common():
        bar = "█" * count
        print(f"  {cost_head:<18} {count:>3}  {bar}")

    print()
    print("--- Sample chunk with full metadata ---")
    sample = next((c for c in all_chunks if c['cost_head'] == 'capex'), all_chunks[0])
    meta_only = {k: v for k, v in sample.items() if k != 'text'}
    print(json.dumps(meta_only, indent=2))
    print(f"  text preview: {sample['text'][:150]}")
else:
    print("⚠️  Run Stage 5 first")

---
## STAGE 7 — Embed + Store in FAISS

**Why sentence-transformers:** Free, runs locally, no API key.  
Model `BAAI/bge-small-en` is lightweight and good for technical English text.

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import pickle

# Load embedding model — downloads ~130MB on first run
print("Loading embedding model (first run downloads ~130MB)...")
embedder = SentenceTransformer('BAAI/bge-small-en')
print("✅ Model loaded")

In [ ]:
def build_vector_store(chunks: list[dict]) -> tuple:
    """
    Embed all chunks and store in FAISS.

    Returns:
        index    → FAISS index (for similarity search)
        metadata → list of chunk dicts (parallel to index)

    How FAISS works:
        - Each chunk text → embedding vector (384 dimensions)
        - FAISS stores all vectors
        - At query time: embed query → find nearest vectors
        - Return chunks whose vectors are closest to query vector
    """
    print(f"Embedding {len(chunks)} chunks...")

    texts = [c['text'] for c in chunks]

    # Embed in batches (faster than one at a time)
    embeddings = embedder.encode(
        texts,
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True   # normalize for cosine similarity
    )

    # Build FAISS index
    # IndexFlatIP = flat index, inner product similarity
    # (= cosine similarity when vectors are normalized)
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatIP(dimension)
    index.add(np.array(embeddings, dtype='float32'))

    print(f"✅ FAISS index built — {index.ntotal} vectors stored")
    return index, chunks   # chunks = metadata store (parallel to index)


# ── TEST: Build vector store ──
if 'all_chunks' in dir() and all_chunks:
    faiss_index, chunk_store = build_vector_store(all_chunks)
else:
    print("⚠️  Run Stage 6 first")

---
## STAGE 8 — Retrieval with Metadata Filtering

This is where everything comes together.  
**Metadata filter first → vector search only on relevant subset.**

In [ ]:
def retrieve(
    query: str,
    faiss_index,
    chunk_store: list[dict],
    top_k: int = 5,
    filters: dict = None
) -> list[dict]:
    """
    Retrieve relevant chunks for a query.

    Steps:
    1. Apply metadata filters to narrow the search space
    2. Embed the query
    3. Search FAISS for nearest vectors
    4. Return top-k chunks with similarity scores

    Filters example:
        {"cost_head": "capex", "filing_year": "FY26"}
    """
    # Step 1: filter chunks by metadata
    if filters:
        candidate_indices = [
            i for i, chunk in enumerate(chunk_store)
            if all(chunk.get(k) == v for k, v in filters.items())
        ]
    else:
        candidate_indices = list(range(len(chunk_store)))

    if not candidate_indices:
        print(f"⚠️  No chunks match filters: {filters}")
        return []

    print(f"Searching {len(candidate_indices)} chunks (filtered from {len(chunk_store)} total)")

    # Step 2: embed query
    query_embedding = embedder.encode(
        [query],
        normalize_embeddings=True
    ).astype('float32')

    # Step 3: get all embeddings for candidate indices
    # Reconstruct vectors for filtered candidates from FAISS
    candidate_embeddings = np.zeros(
        (len(candidate_indices), faiss_index.d), dtype='float32'
    )
    for i, idx in enumerate(candidate_indices):
        faiss_index.reconstruct(idx, candidate_embeddings[i])

    # Step 4: compute similarity scores
    scores = np.dot(candidate_embeddings, query_embedding.T).flatten()

    # Get top-k
    top_k_local = min(top_k, len(candidate_indices))
    top_indices_local = np.argsort(scores)[::-1][:top_k_local]

    results = []
    for local_idx in top_indices_local:
        global_idx = candidate_indices[local_idx]
        chunk = chunk_store[global_idx].copy()
        chunk["similarity_score"] = float(scores[local_idx])
        results.append(chunk)

    return results


# ── TEST: Run queries ──
if 'faiss_index' in dir():

    print("=" * 60)
    print("TEST 1: Query without filters")
    print("=" * 60)
    results = retrieve(
        query="What is the capital expenditure for FY26?",
        faiss_index=faiss_index,
        chunk_store=chunk_store,
        top_k=3
    )
    for i, r in enumerate(results):
        print(f"\nResult {i+1} | Score: {r['similarity_score']:.3f} | "
              f"Page {r['page_number']} | cost_head: {r['cost_head']}")
        print(f"  {r['text'][:200]}")

    print()
    print("=" * 60)
    print("TEST 2: Query WITH metadata filter (capex only)")
    print("=" * 60)
    results_filtered = retrieve(
        query="What is the capital expenditure for FY26?",
        faiss_index=faiss_index,
        chunk_store=chunk_store,
        top_k=3,
        filters={"cost_head": "capex"}
    )
    for i, r in enumerate(results_filtered):
        print(f"\nResult {i+1} | Score: {r['similarity_score']:.3f} | "
              f"Page {r['page_number']} | cost_head: {r['cost_head']}")
        print(f"  {r['text'][:200]}")
else:
    print("⚠️  Run Stage 7 first")

---
## STAGE 9 — Save & Load the Pipeline

So you don't have to re-embed every time you restart the notebook.

In [ ]:
import os

SAVE_DIR = "data/vector_store"
os.makedirs(SAVE_DIR, exist_ok=True)

def save_pipeline(faiss_index, chunk_store: list[dict], save_dir: str):
    """Save FAISS index + chunk metadata to disk."""
    faiss.write_index(faiss_index, f"{save_dir}/index.faiss")
    with open(f"{save_dir}/chunks.pkl", "wb") as f:
        pickle.dump(chunk_store, f)
    print(f"✅ Saved to {save_dir}/")
    print(f"   index.faiss  — {os.path.getsize(f'{save_dir}/index.faiss'):,} bytes")
    print(f"   chunks.pkl   — {os.path.getsize(f'{save_dir}/chunks.pkl'):,} bytes")


def load_pipeline(save_dir: str) -> tuple:
    """Load FAISS index + chunk metadata from disk."""
    faiss_index = faiss.read_index(f"{save_dir}/index.faiss")
    with open(f"{save_dir}/chunks.pkl", "rb") as f:
        chunk_store = pickle.load(f)
    print(f"✅ Loaded from {save_dir}/")
    print(f"   Vectors: {faiss_index.ntotal} | Chunks: {len(chunk_store)}")
    return faiss_index, chunk_store


# Save
if 'faiss_index' in dir():
    save_pipeline(faiss_index, chunk_store, SAVE_DIR)

    # Verify by loading back
    print()
    print("Verifying load...")
    loaded_index, loaded_chunks = load_pipeline(SAVE_DIR)
    print("✅ Save/Load verified")
else:
    print("⚠️  Run Stage 7 first")

---
## STAGE 10 — Full Pipeline Test (End to End)

Run this cell to test the **complete flow** from PDF → answer.

In [ ]:
def run_full_pipeline(pdf_path: str, doc_metadata: dict, max_pages: int = None):
    """
    End-to-end pipeline:
    PDF → pages → chunks → metadata → embeddings → FAISS
    """
    print("STEP 1: Loading PDF")
    print("-" * 40)
    pages = load_pdf(pdf_path, doc_metadata, max_pages=max_pages)

    print("\nSTEP 2: Chunking pages")
    print("-" * 40)
    all_chunks = []
    for page in pages:
        all_chunks.extend(chunk_page(page))
    print(f"Total chunks: {len(all_chunks)}")

    print("\nSTEP 3: Adding cost_head metadata")
    print("-" * 40)
    all_chunks = add_metadata_to_chunks(all_chunks)
    from collections import Counter
    print(Counter(c['cost_head'] for c in all_chunks).most_common(5))

    print("\nSTEP 4: Building vector store")
    print("-" * 40)
    index, store = build_vector_store(all_chunks)

    return index, store


# ── Run complete pipeline ──
if Path(PDF_PATH).exists():
    faiss_index, chunk_store = run_full_pipeline(
        PDF_PATH,
        DOC_METADATA,
        max_pages=30   # increase this once you're confident it works
    )
    save_pipeline(faiss_index, chunk_store, SAVE_DIR)
else:
    print("⚠️  Set PDF_PATH in Cell 3 first")

---
## STAGE 11 — Your Test Queries

Try your own queries here. Experiment with different filters.

In [ ]:
# ── Try your own queries below ──

MY_QUERIES = [
    # (query_text, filters_dict)
    ("What is ARR for FY26?",                         None),
    ("What is projected capitalization?",             {"cost_head": "capex"}),
    ("Why did interest cost increase?",               {"cost_head": "interest"}),
    ("What tariff hike is requested?",                {"cost_head": "tariff"}),
    ("Employee cost trend across years",              {"cost_head": "employee_cost"}),
]

if 'faiss_index' in dir():
    for query, filters in MY_QUERIES:
        print(f"\n{'='*60}")
        print(f"QUERY  : {query}")
        print(f"FILTERS: {filters}")
        print("=" * 60)

        results = retrieve(
            query=query,
            faiss_index=faiss_index,
            chunk_store=chunk_store,
            top_k=2,
            filters=filters
        )

        for i, r in enumerate(results):
            print(f"\n  Result {i+1}")
            print(f"  Score     : {r['similarity_score']:.3f}")
            print(f"  Page      : {r['page_number']}")
            print(f"  cost_head : {r['cost_head']}")
            print(f"  chunk_type: {r['chunk_type']}")
            print(f"  Text      : {r['text'][:250]}")
else:
    print("⚠️  Run Stage 10 first")

---
## What's next — Week 2 preview

Now that retrieval works, Week 2 adds:

| What | Why |
|------|-----|
| LLM answer generation | Turn retrieved chunks → actual answers |
| Citation engine | Every answer must cite page + section |
| Multi-document retrieval | Query across petition + tariff order together |
| Better table handling | `camelot` for complex multi-row tables |

**Before Week 2 — your homework:**
1. Run this notebook on your actual JUSNL petition PDF
2. Look at the cost_head distribution — are the mappings accurate?
3. Try a query that gives a wrong result — note which stage failed
4. Ask yourself: why might the keyword mapper tag a chunk as `other` even if it's about capex?

---
*Power Regulatory RAG — Week 1 complete*